# docpilot — Colab Setup

Production RAG system running directly in Google Colab.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Add secrets (🔑 icon in left sidebar):
   - `GROQ_API_KEY` — get free at [console.groq.com](https://console.groq.com)
   - `NGROK_TOKEN` — get free at [ngrok.com](https://ngrok.com)

**Services started by this notebook:**
| Service | Port | Purpose |
|---------|------|---------|
| FastAPI | 8000 | REST API + RAG pipeline |
| Streamlit | 8501 | Chat UI |
| Qdrant | 6333 | Vector store |


In [ ]:
# Cell 1 — Check environment
import subprocess

result = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
print('=== Disk Space ===')
print(result.stdout)

result = subprocess.run(['free', '-h'], capture_output=True, text=True)
print('=== RAM ===')
print(result.stdout)

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print('=== GPU ===')
print(result.stdout if result.returncode == 0 else 'No GPU detected')

In [ ]:
# Cell 2 — Install system dependencies
!apt-get update -qq
!apt-get install -y -qq poppler-utils curl wget
print('✅ System deps installed')

In [ ]:
# Cell 3 — Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
!uv --version
print('✅ uv installed')

In [ ]:
# Cell 4 — Clone repo and install backend deps
!git clone https://github.com/shawnk1188/docpilot.git
%cd docpilot

%cd backend
!uv python install 3.12
!uv python pin 3.12
!uv sync
%cd ..
print('✅ Backend deps installed')

In [ ]:
# Cell 5 — Download and start Qdrant (local storage)
import subprocess, time, requests

!wget -q https://github.com/qdrant/qdrant/releases/download/v1.17.0/qdrant-x86_64-unknown-linux-musl.tar.gz
!tar -xzf qdrant-x86_64-unknown-linux-musl.tar.gz

qdrant_proc = subprocess.Popen(
    ['./qdrant'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print('Waiting for Qdrant...')
for i in range(30):
    try:
        r = requests.get('http://localhost:6333/')
        if r.status_code == 200:
            print(f'✅ Qdrant ready: v{r.json()["version"]}')
            break
    except:
        time.sleep(1)
        print(f'  attempt {i+1}/30...')

In [ ]:
# Cell 6 — Configure environment from Colab Secrets
import os
from google.colab import userdata

# Read secrets
groq_key    = userdata.get('GROQ_API_KEY')
ngrok_token = userdata.get('NGROK_TOKEN')

print(f'✅ Secrets loaded')
print(f'   GROQ_API_KEY : {groq_key[:8]}...{groq_key[-4:]}')
print(f'   NGROK_TOKEN  : {ngrok_token[:8]}...')

# LLM
os.environ['LLM_PROVIDER']      = 'groq'
os.environ['GROQ_API_KEY']      = groq_key
os.environ['GROQ_BASE_URL']     = 'https://api.groq.com/openai/v1'
os.environ['GROQ_MODEL']        = 'llama-3.1-8b-instant'
os.environ['OLLAMA_BASE_URL']   = 'http://localhost:11434'
os.environ['OLLAMA_MODEL']      = 'tinyllama'

# Qdrant
os.environ['QDRANT_HOST']       = 'localhost'
os.environ['QDRANT_PORT']       = '6333'
os.environ['QDRANT_COLLECTION'] = 'docpilot'

# Embedding
os.environ['EMBEDDING_MODEL']   = 'all-MiniLM-L6-v2'
os.environ['VECTOR_DIM']        = '384'

# Chunking
os.environ['CHUNK_SIZE']        = '600'
os.environ['CHUNK_OVERLAP']     = '100'

# Retrieval
os.environ['RETRIEVAL_TOP_K']   = '5'
os.environ['FETCH_K']           = '20'
os.environ['RERANKER_ENABLED']  = 'true'
os.environ['RERANKER_MODEL']    = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
os.environ['BACKEND_URL']       = 'http://localhost:8000'

print('✅ Environment configured')
print(f'   LLM    : {os.environ["LLM_PROVIDER"]} / {os.environ["GROQ_MODEL"]}')
print(f'   Qdrant : {os.environ["QDRANT_HOST"]}:{os.environ["QDRANT_PORT"]}')

In [ ]:
# Cell 7 — Start FastAPI backend
import glob, subprocess, time, requests, os

# Find uv-managed Python 3.12
python_paths = glob.glob(
    '/root/.local/share/uv/python/cpython-3.12.*/bin/python'
)

if python_paths:
    python_bin = python_paths[0]
    cmd = [python_bin, '-m', 'uvicorn', 'app.main:app',
           '--host', '0.0.0.0', '--port', '8000']
else:
    cmd = ['uv', 'run', 'uvicorn', 'app.main:app',
           '--host', '0.0.0.0', '--port', '8000']

print(f'Starting: {" ".join(cmd)}')

fastapi_proc = subprocess.Popen(
    cmd,
    cwd='/content/docpilot/backend',
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print('Waiting for FastAPI...')
for i in range(30):
    try:
        r = requests.get('http://localhost:8000/health')
        if r.status_code == 200:
            data = r.json()
            print(f'✅ FastAPI ready')
            print(f'   Status : {data["status"]}')
            print(f'   Qdrant : {data["qdrant"]}')
            print(f'   LLM    : {data["llm_model"]}')
            break
    except:
        time.sleep(1)
        print(f'  attempt {i+1}/30...')
else:
    print('❌ FastAPI failed — logs:')
    out, err = fastapi_proc.communicate(timeout=3)
    print(err.decode())

In [ ]:
# Cell 8 — Expose via ngrok
!pip install pyngrok -q
from pyngrok import ngrok
from google.colab import userdata
import os

ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))

api_tunnel = ngrok.connect(8000)
api_url    = api_tunnel.public_url
os.environ['BACKEND_URL'] = api_url

print(f'✅ FastAPI  → {api_url}')
print(f'   API docs → {api_url}/docs')

In [ ]:
# Cell 9 — Start Streamlit frontend
import subprocess, time, os
from pyngrok import ngrok

# Install frontend deps
!pip install streamlit httpx -q

streamlit_proc = subprocess.Popen(
    ['python', '-m', 'streamlit', 'run', 'frontend/app.py',
     '--server.port', '8501',
     '--server.address', '0.0.0.0',
     '--server.headless', 'true',
     '--browser.gatherUsageStats', 'false'],
    cwd='/content/docpilot',
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

time.sleep(5)

ui_tunnel = ngrok.connect(8501)
print(f'✅ Streamlit UI → {ui_tunnel.public_url}')
print('')
print('Open the Streamlit URL above to use docpilot')

In [ ]:
# Cell 10 — Ingest a document
import requests, json
from google.colab import files

print('Upload a PDF to ingest:')
uploaded = files.upload()

for filename, content in uploaded.items():
    print(f'Ingesting {filename}...')
    resp = requests.post(
        'http://localhost:8000/api/ingest',
        files={'file': (filename, content, 'application/pdf')},
        timeout=120,
    )
    data = resp.json()
    print(json.dumps(data, indent=2))

In [ ]:
# Cell 11 — Ask questions
import requests, json

def ask(question, top_k=5):
    resp = requests.post(
        'http://localhost:8000/api/query',
        json={'question': question, 'top_k': top_k},
        timeout=60,
    )
    data = resp.json()
    print(f'\n{"="*60}')
    print(f'Q: {question}')
    print(f'{"="*60}')
    print(f'A: {data["answer"]}')
    print(f'\n📎 Sources ({len(data["sources"])} chunks):')
    for i, src in enumerate(data['sources'], 1):
        page  = f'p.{src["page_number"]}' if src['page_number'] else 'n/a'
        score = src['score']
        icon  = '🟢' if score > 0.7 else '🟡' if score > 0.5 else '🔴'
        print(f'  [{i}] {icon} {src["source_file"]} {page} — {score}')
    print(f'Model: {data["model"]}')
    return data

# Test queries
ask('What is this document about?')
ask('What is a probability mass function?')
ask('What is the capital of France?')  # should abstain

In [ ]:
# Cell 12 — Collection stats
import requests, json

resp = requests.get('http://localhost:8000/api/stats')
print(json.dumps(resp.json(), indent=2))

In [ ]:
# Cell 13 — Stop all services
def stop_all():
    fastapi_proc.terminate()
    streamlit_proc.terminate()
    qdrant_proc.terminate()
    from pyngrok import ngrok
    ngrok.kill()
    print('✅ All services stopped')

# Uncomment to stop:
# stop_all()